In [1]:
import pickle 
import functools
import networkx as nx
import numpy as np

# package(s) related to the simulation (creating the vessel, running the simulation)
import xarray as xr
import datetime
import simpy
import opentnsim
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.plotutils import generate_vessel_gantt_chart
from opentnsim.graph import mixins as graph_module

# package(s) needed for inspecting the output
import pandas as pd

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'data/d-osp/gtsm'

In [3]:
ds = xr.open_dataset(src_dir / 'filtered-currents.nc')

In [4]:
with open(src_dir / 'di_graph_currents.pickle', 'rb') as f:
    G = pickle.load(f)

In [5]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        opentnsim.core.Identifiable, # allows to give the object a name and a random ID,
        opentnsim.core.Movable,      # allows the object to move, with a fixed speed, while logging this activity
    ), 
    {}
)

In [6]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel. 
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [7]:
simulation_start = datetime.datetime(2024, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

In [8]:
def __compute_weight(
    origin, target, dictionary_edge, ship_velocity
):
    # order classes from smallest to largest
    # if dictionary_edge["is_border"]:
    #     return dictionary_edge["length_m"]
            
    edge_length = dictionary_edge["length"]
    edge_current = dictionary_edge['Info']["Current"]
    ship_velocity += edge_current

    taken_time = edge_length / ship_velocity
    return taken_time


def path_corrected(
    graph, origin, destination, ship_velocity
):
    """find a path restricted to allowed cemt classes

    Parameters
    ----------
    graph : networkx.Graph
        graph in which to find a path. graph edges should have information 'cemt'.
    origin : str
        origin node id
    destination : str
        destination node id
    ship_cemt_classe : str
        cemt class of the ship.
    """
    # define order of cemt classes

    # create function to compute weights for this ship
    compute_weight = functools.partial(
        __compute_weight,
        ship_velocity=ship_velocity
    )

    # find the path
    path = nx.dijkstra_path(graph, origin, destination, weight=compute_weight)
    return path

In [9]:
env.graph = G

# create vessel from a dict 
forward_path = path_corrected(graph=G, origin=126, destination=183, ship_velocity=3)
backward_path = path_corrected(graph=G, origin=183, destination=126, ship_velocity=3)

# Combine them, skipping the first node of the backward path to avoid duplication
path = forward_path + backward_path[1:]


data_vessel = {
    "env": env,                                       # needed for simpy simulation
    "name": "Vessel",                                 # required by Identifiable
    "geometry": env.graph.nodes[forward_path[0]]['geometry'], # required by Locatable
    "route": forward_path,                                    # required by Routeable
    "v": 3,                                           # required by Movable, 1 m/s to check if the distance is covered in the expected time
}  # 

# create an instance of the Vessel class using the input dict data_vessel
vessel = Vessel(**data_vessel)

# start the simulation
env.process(mission(env, vessel))
env.run()

In [10]:
ds.time < datetime.datetime.fromtimestamp(1704123324.171281).strftime("%Y-%m-%d %H:%M:%S")

UFuncTypeError: ufunc 'less' did not contain a loop with signature matching types (<class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.StrDType'>) -> <class 'numpy.dtypes.BoolDType'>

In [11]:
ds.time

<xarray.DataArray 'time' (time: 241)> Size: 2kB
array(['2025-11-08T12:00:00.000000000', '2025-11-08T13:00:00.000000000',
       '2025-11-08T14:00:00.000000000', ..., '2025-11-18T10:00:00.000000000',
       '2025-11-18T11:00:00.000000000', '2025-11-18T12:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 2kB 2025-11-08T12:00:00 ... 2025-11-18T12:...

In [12]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel.logbook)

print("'{}' logbook data:".format(vessel.name))  
print('')

display(df)

trip_distance = graph_module.calculate_distance_along_path(G, vessel.route)
trip_duration = datetime.timedelta.total_seconds(vessel.logbook[-1]['Timestamp'] - vessel.logbook[0]['Timestamp'])

print("'{}' travelled a distance of {:.1f} meters".format(vessel.name, trip_distance))
print("'{}' took {:.1f} seconds to arrive at its destination".format(vessel.name, trip_duration))  
print("'{}' travelled at an average speed of {:.1f} meters per second".format(vessel.name, trip_distance/trip_duration))
print('')
print('')

'Vessel' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 126 to node 192 start,2024-01-01 00:00:00.000000,0.000000,POINT (4.088430626976463 52.01736708594974)
1,Sailing from node 126 to node 192 stop,2024-01-01 01:21:04.438099,13470.069038,POINT (4.189314201519081 52.0841832301647)
2,Sailing from node 192 to node 287 start,2024-01-01 01:21:04.438099,13470.069038,POINT (4.189314201519081 52.0841832301647)
3,Sailing from node 192 to node 287 stop,2024-01-01 02:38:31.210231,26205.902646,POINT (4.277204815842396 52.15742540801211)
4,Sailing from node 287 to node 485 start,2024-01-01 02:38:31.210231,26205.902646,POINT (4.277204815842396 52.15742540801211)
5,Sailing from node 287 to node 485 stop,2024-01-01 03:59:23.938136,39265.658520,POINT (4.37058327419094 52.2284460088555)
6,Sailing from node 485 to node 6 start,2024-01-01 03:59:23.938136,39265.658520,POINT (4.37058327419094 52.2284460088555)
7,Sailing from node 485 to node 6 stop,2024-01-01 03:59:23.977145,39265.763497,POINT (4.4204069239239345 52.31153174777205)
8,Sailing from node 6 to node 147 start,2024-01-01 03:59:23.977145,39265.763497,POINT (4.4204069239239345 52.31153174777205)
9,Sailing from node 6 to node 147 stop,2024-01-01 04:57:40.009316,48449.378579,POINT (4.4357723612757685 52.39258600920478)


'Vessel' travelled a distance of 125978.1 meters
'Vessel' took 36360.7 seconds to arrive at its destination
'Vessel' travelled at an average speed of 3.5 meters per second




In [18]:
env.graph = G

# create vessel from a dict 
forward_path = nx.dijkstra_path(G, source=126, target=183)
backward_path = nx.dijkstra_path(G, source=183, target=126)

# Combine them, skipping the first node of the backward path to avoid duplication
path = forward_path + backward_path[1:]


data_vessel = {
    "env": env,                                       # needed for simpy simulation
    "name": "Vessel",                                 # required by Identifiable
    "geometry": env.graph.nodes[path[0]]['geometry'], # required by Locatable
    "route": path,                                    # required by Routeable
    "v": 3,                                           # required by Movable, 1 m/s to check if the distance is covered in the expected time
}  # 

# create an instance of the Vessel class using the input dict data_vessel
vessel = Vessel(**data_vessel)

# start the simulation
env.process(mission(env, vessel))
env.run()

# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel.logbook)

print("'{}' logbook data:".format(vessel.name))  
print('')

display(df)

trip_distance = graph_module.calculate_distance_along_path(G, vessel.route)
trip_duration = datetime.timedelta.total_seconds(vessel.logbook[-1]['Timestamp'] - vessel.logbook[0]['Timestamp'])

print("'{}' travelled a distance of {:.1f} meters".format(vessel.name, trip_distance))
print("'{}' took {:.1f} seconds to arrive at its destination".format(vessel.name, trip_duration))  
print("'{}' travelled at an average speed of {:.1f} meters per second".format(vessel.name, trip_distance/trip_duration))
print('')
print('')

'Vessel' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 126 to node 192 start,2024-01-01 16:35:24.171281,0.000000,POINT (4.088430626976463 52.01736708594974)
1,Sailing from node 126 to node 192 stop,2024-01-01 17:56:28.609380,13470.069038,POINT (4.189314201519081 52.0841832301647)
2,Sailing from node 192 to node 68 start,2024-01-01 17:56:28.609380,13470.069038,POINT (4.189314201519081 52.0841832301647)
3,Sailing from node 192 to node 68 stop,2024-01-01 17:56:28.644554,13470.164868,POINT (4.155561347809885 52.18053143400781)
4,Sailing from node 68 to node 194 start,2024-01-01 17:56:28.644554,13470.164868,POINT (4.155561347809885 52.18053143400781)
5,Sailing from node 68 to node 194 stop,2024-01-01 19:41:24.965093,29571.645740,POINT (4.186318271939377 52.32186557069249)
6,Sailing from node 194 to node 195 start,2024-01-01 19:41:24.965093,29571.645740,POINT (4.186318271939377 52.32186557069249)
7,Sailing from node 194 to node 195 stop,2024-01-01 21:38:43.478544,48253.261815,POINT (4.083007841015341 52.45411701430916)
8,Sailing from node 195 to node 256 start,2024-01-01 21:38:43.478544,48253.261815,POINT (4.083007841015341 52.45411701430916)
9,Sailing from node 195 to node 256 stop,2024-01-02 00:05:50.465036,70643.770923,POINT (4.254876451772548 52.558603474870566)


'Vessel' travelled a distance of 299146.4 meters
'Vessel' took 70588.1 seconds to arrive at its destination
'Vessel' travelled at an average speed of 4.2 meters per second


